In [ ]:
import least_entropy as le
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import os
import numpy as np

In [ ]:
def procesar_archivos(carpeta, x_parts=8, y_parts=8, p0=0.4, p1=500, p_num=10000, plot_individual=False, plot_s=False):
    """
    Procesa los primeros N archivos de la carpeta
    
    Parámetros:
    -----------
    carpeta : str
        Ruta a la carpeta con los archivos .dat
    N : int
        Número de archivos a procesar
    p0, p1, np : parámetros para find_best_period
    plot_individual : bool
        Si True, muestra la curva de luz de cada estrella
    """
    
    # Obtener lista de archivos .dat y ordenarlos numéricamente
    #archivos = [f for f in os.listdir(carpeta) if f.endswith('.dat')]
    archivos = [f for f in os.listdir(carpeta)]
    
    # Limitar a los primeros N
    #archivos = archivos[:N]
    
    print(f"Procesando {len(archivos)} archivos de {carpeta}")
    
    # Lista para guardar resultados
    resultados = []
    
    for archivo in tqdm(archivos, desc="Procesando estrellas"):
        try:
            # Cargar datos
            ruta_completa = os.path.join(carpeta, archivo)
            test_data = pd.read_csv(ruta_completa, sep=" ", lineterminator="\n", 
                                    names=("t", "u", "inc_u"))
            
            # Encontrar período
            best_period, min_entropy, final_phases, second_phases = le.find_best_period(
                test_data, 
                plot_entropies=plot_s,
                p0=p0,
                p1=p1,
                p_num=p_num,
                t_parts=x_parts,
                u_parts=y_parts
            )
            
            # Guardar resultado
            resultados.append({
                'archivo': archivo,
                'id': archivo.replace('.dat', ''),
                'best_period': best_period,
                'min_entropy': min_entropy,
                'n_puntos': len(test_data)
            })
            
            # Plot individual si se solicita
            if plot_individual:
                plt.figure(figsize=(8, 4))
                plt.scatter(final_phases, test_data["u"], s=5, alpha=0.7)
                plt.scatter(second_phases, test_data["u"], s=5, alpha=0.7)
                plt.xlabel('Fase')
                plt.ylabel('Magnitud')
                plt.title(f'Estrella {archivo} - Período = {best_period:.4f} días')
                plt.gca().invert_yaxis()  # Para magnitudes
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
                
        except Exception as e:
            print(f"Error procesando {archivo}: {e}")
            resultados.append({
                'archivo': archivo,
                'id': archivo.replace('.dat', ''),
                'best_period': None,
                'min_entropy': None,
                'n_puntos': None,
                'error': str(e)
            })
    
    # Crear DataFrame con resultados
    df_resultados = pd.DataFrame(resultados)
    
    # Mostrar resumen
    print("\n" + "="*60)
    print("RESUMEN DE RESULTADOS")
    print("="*60)
    for idx, row in df_resultados.iterrows():
        if row['best_period'] is not None:
            print(f"{row['id']}: Período = {row['best_period']:.6f} días, "
                  f"Entropía = {row['min_entropy']:.4f}, Puntos = {row['n_puntos']}")
        else:
            print(f"{row['id']}: ERROR - {row.get('error', 'Desconocido')}")
    
    return df_resultados

In [ ]:
# CEFEIDAS
if __name__ == "__main__":
    # Procesar primeros 5 archivos
    resultados = procesar_archivos(
        carpeta="data/cefeidas",
        p0=0.4,
        p1=200,
        p_num=5000,  # Menos puntos para pruebas rápidas
        plot_individual=True,  # Muestra cada curva de luz
        x_parts=12,
        y_parts=7
    )
    
    # Guardar resultados a CSV
    resultados.to_csv("resultados_periodos_cefeidas.csv", index=False)
    print(f"\nResultados guardados en 'resultados_periodos.csv'")

In [ ]:
# RR LYRAE
if __name__ == "__main__":
    # Procesar primeros 5 archivos
    resultados = procesar_archivos(
        carpeta="data/rr_lyrae",
        p0=0.15,
        p1=1.2,
        p_num=8000,  
        plot_individual=True,  # Muestra cada curva de luz
        x_parts=12,
        y_parts=7,
        plot_s=True
    )
    
    # Guardar resultados a CSV
    resultados.to_csv("resultados_periodos_rrlyrae.csv", index=False)
    print(f"\nResultados guardados en 'resultados_periodos.csv'")

In [ ]:
# BINARIAS ECLIPSANTES
if __name__ == "__main__":
    # Procesar primeros 5 archivos
    resultados = procesar_archivos(
        carpeta="data/eclipsing",
        p0=0.4,
        p1=500,
        p_num=5000,  
        plot_individual=True,  # Muestra cada curva de luz
        x_parts=17,
        y_parts=5
    )
    
    # Guardar resultados a CSV
    resultados.to_csv("resultados_periodos_rrlyrae.csv", index=False)
    print(f"\nResultados guardados en 'resultados_periodos.csv'")